In [1]:
# ============================================================
# NOTEBOOK 06 — FULL RAG PIPELINE END TO END
# ============================================================
# This notebook combines ALL previous steps into one
# clean, readable pipeline.
#
# THE FULL FLOW:
#
#   PDF Document
#       ↓
#   STEP 1: INGESTION    — load the PDF
#       ↓
#   STEP 2: CHUNKING     — split into small pieces
#       ↓
#   STEP 3: VECTORIZE    — convert chunks to numbers
#       ↓
#   STEP 4: STORE        — save in ChromaDB
#       ↓
#   STEP 5: RETRIEVE     — find relevant chunks
#       ↓
#   STEP 6: PROMPT       — build instruction for LLM
#       ↓
#   STEP 7: GENERATE     — LLM writes the answer
#       ↓
#   STEP 8: OUTPUT       — clean answer returned to user
#
# After this notebook we convert everything into
# proper Python modules in the src/ folder!
# ============================================================

print("="*60)
print("  DocuMind — Full RAG Pipeline")
print("="*60)

  DocuMind — Full RAG Pipeline


In [2]:
# ============================================================
# ALL IMPORTS
# ============================================================

import os

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

# PDF loading
from langchain_community.document_loaders import PyPDFLoader

# Text splitting
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Embeddings — runs locally, no API needed
from langchain_huggingface import HuggingFaceEmbeddings

# Vector database
from langchain_community.vectorstores import Chroma

# Local LLM — runs on your machine
from transformers import pipeline

import requests

print("All libraries imported!")

All libraries imported!


In [3]:
# ============================================================
# CONFIGURATION — ALL SETTINGS IN ONE PLACE
# ============================================================
# Keeping all settings here makes it easy to change later
# For example: swap the PDF, change chunk size, change model

# Path to your document
PDF_PATH = "../data/documents/attention_is_all_you_need.pdf"

# Where to save the vector database
CHROMA_PATH = "../.chroma"

# Chunking settings
CHUNK_SIZE = 500      # characters per chunk
CHUNK_OVERLAP = 50    # overlap between chunks

# Embedding model — runs locally, completely free
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# LLM model — runs locally, completely free
LLM_MODEL = "google/flan-t5-small"

# How many chunks to retrieve per question
TOP_K_CHUNKS = 3

print("Configuration loaded!")
print(f"  Document    : {PDF_PATH}")
print(f"  Chunk size  : {CHUNK_SIZE}")
print(f"  Embedding   : {EMBEDDING_MODEL}")
print(f"  LLM         : {LLM_MODEL}")
print(f"  Top k chunks: {TOP_K_CHUNKS}")

Configuration loaded!
  Document    : ../data/documents/attention_is_all_you_need.pdf
  Chunk size  : 500
  Embedding   : sentence-transformers/all-MiniLM-L6-v2
  LLM         : google/flan-t5-small
  Top k chunks: 3


In [4]:
# ============================================================
# STEP 1 & 2: INGESTION AND CHUNKING
# ============================================================

print("STEP 1: Loading document...")

# Load the PDF — each page becomes a Document object
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()
print(f"  Loaded {len(documents)} pages")

print("\nSTEP 2: Chunking document...")

# Split pages into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    add_start_index=True
)

# Split all pages into chunks
chunks = text_splitter.split_documents(documents)
print(f"  Created {len(chunks)} chunks from {len(documents)} pages")
print(f"  Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")

STEP 1: Loading document...
  Loaded 15 pages

STEP 2: Chunking document...
  Created 94 chunks from 15 pages
  Average chunk size: 429 characters


In [5]:
# ============================================================
# STEP 3 & 4: VECTORIZATION AND STORAGE
# ============================================================

print("STEP 3: Loading embedding model...")

# Load the local embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
print(f"  Embedding model loaded: {EMBEDDING_MODEL}")

print("\nSTEP 4: Storing chunks in ChromaDB...")

# Check if vector store already exists on disk
# If yes — load it (saves time, no re-embedding needed)
# If no  — create it fresh
if os.path.exists(CHROMA_PATH):
    print("  Found existing vector store — loading from disk...")
    vectorstore = Chroma(
        persist_directory=CHROMA_PATH,
        embedding_function=embedding_model
    )
else:
    print("  No existing vector store — creating fresh...")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=CHROMA_PATH
    )

print(f"  Vector store ready: {vectorstore._collection.count()} chunks stored")

STEP 3: Loading embedding model...
  Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2

STEP 4: Storing chunks in ChromaDB...
  Found existing vector store — loading from disk...


/var/folders/d0/ymvkftpj5hzdbp0bfg1vxwvr0000gn/T/ipykernel_67375/255494675.py:22: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


  Vector store ready: 94 chunks stored


In [6]:
# ============================================================
# STEP 5: RETRIEVAL
# ============================================================

print("STEP 5: Setting up retriever...")

# Create retriever from vector store
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K_CHUNKS}
)

print(f"  Retriever ready — will return top {TOP_K_CHUNKS} chunks per question")

STEP 5: Setting up retriever...
  Retriever ready — will return top 3 chunks per question


In [7]:
# ============================================================
# STEP 6 & 7: PROMPT BUILDING AND GENERATION
# ============================================================

print("STEP 6 & 7: Loading local LLM...")

# Load the local model
local_llm = pipeline(
    "text2text-generation",
    model=LLM_MODEL,
    max_new_tokens=200
)

def build_prompt(context, question):
    """Build the prompt by combining context and question."""
    return f"""Answer the question based only on the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{question}

Answer:"""

def ask(question):
    """
    Full RAG pipeline in one function.
    Takes a question and returns a grounded answer.
    """

    # STEP 5 — retrieve relevant chunks
    retrieved_chunks = retriever.invoke(question)

    # STEP 6 — format chunks into context string
    context = "\n\n".join(
        chunk.page_content for chunk in retrieved_chunks
    )

    # STEP 6 — build the prompt
    prompt = build_prompt(context, question)

    # STEP 7 — generate answer with local LLM
    result = local_llm(prompt)
    answer = result[0]["generated_text"]

    return answer, retrieved_chunks

print(f"  LLM loaded: {LLM_MODEL}")
print("  Pipeline ready!")

STEP 6 & 7: Loading local LLM...


Device set to use mps:0


  LLM loaded: google/flan-t5-small
  Pipeline ready!


In [8]:
# ============================================================
# STEP 8: OUTPUT — ASK QUESTIONS AND GET ANSWERS
# ============================================================

print("="*60)
print("  DocuMind is ready! Asking questions...")
print("="*60)

questions = [
    "What is the attention mechanism?",
    "What is multi-head attention?",
    "What datasets were used for training?",
    "What were the BLEU scores on translation tasks?",
    "What is the role of positional encoding?"
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"QUESTION: {question}")
    print(f"{'='*60}")

    # Get answer and retrieved chunks
    answer, chunks_used = ask(question)

    # Show which pages were used
    pages_used = [str(c.metadata.get('page', '?')) for c in chunks_used]
    print(f"Sources  : pages {', '.join(pages_used)}")
    print(f"Answer   : {answer}")

  DocuMind is ready! Asking questions...

QUESTION: What is the attention mechanism?
Sources  : pages 12, 14, 1
Answer   : relating different positions of a single sequence in order to compute a representation of the sequence

QUESTION: What is multi-head attention?
Sources  : pages 1, 4, 14
Answer   : allows the model to jointly attend to information from different representation subspaces at different positions

QUESTION: What datasets were used for training?
Sources  : pages 6, 8, 6
Answer   : WMT 2014 English-French dataset consisting of 36M sentences and split tokens into a 32000 word-piece vocabulary [38]. Sentence pairs were batched together by approximate sequence length. Each training batch contained a set of sentence pairs containing approximately 25000 source tokens and 25000 models have not been able to attain state-of-the-art results in small-data regimes [37]. We trained a 4-layer transformer with dmodel = 1024on the Wall Street Journal (WSJ) portion of the Penn Treebank 

In [9]:
# ============================================================
# BONUS: FULL TRANSPARENCY — SEE EVERY STEP FOR ONE QUESTION
# ============================================================

question = "What is positional encoding?"

print(f"QUESTION: {question}")
print()

# STEP 1 — retrieve
retrieved = retriever.invoke(question)
print(f"RETRIEVED CHUNKS ({len(retrieved)}):")
for i, chunk in enumerate(retrieved):
    print(f"\n  Chunk {i+1} — page {chunk.metadata.get('page','?')}:")
    print(f"  {chunk.page_content[:200]}...")

# STEP 2 — build context
context = "\n\n".join(c.page_content for c in retrieved)
print(f"\nCONTEXT LENGTH: {len(context)} characters")

# STEP 3 — build prompt
prompt = build_prompt(context, question)
print(f"\nFULL PROMPT:")
print(prompt)

# STEP 4 — generate
result = local_llm(prompt)
answer = result[0]["generated_text"]
print(f"\nFINAL ANSWER:")
print(answer)

QUESTION: What is positional encoding?

RETRIEVED CHUNKS (3):

  Chunk 1 — page 5:
  bottoms of the encoder and decoder stacks. The positional encodings have the same dimension dmodel
as the embeddings, so that the two can be summed. There are many choices of positional encodings,
lea...

  Chunk 2 — page 4:
  [38, 2, 9].
• The encoder contains self-attention layers. In a self-attention layer all of the keys, values
and queries come from the same place, in this case, the output of the previous layer in the
...

  Chunk 3 — page 5:
  Convolutional O(k · n · d2) O(1) O(logk(n))
Self-Attention (restricted) O(r · n · d) O(1) O(n/r)
3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the mod...

CONTEXT LENGTH: 1389 characters

FULL PROMPT:
Answer the question based only on the context below.
If the answer is not in the context, say "I don't know".

Context:
bottoms of the encoder and decoder stacks. The positional encodings have the same dimensio

In [10]:
# ============================================================
# SUMMARY — THE COMPLETE RAG PIPELINE
# ============================================================

print("""
╔══════════════════════════════════════════════════════════╗
║           DocuMind RAG Pipeline — Complete!              ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  STEP 1  Ingestion      PyPDFLoader                      ║
║  STEP 2  Chunking       RecursiveCharacterTextSplitter   ║
║  STEP 3  Vectorization  HuggingFace all-MiniLM-L6-v2     ║
║  STEP 4  Storage        ChromaDB                         ║
║  STEP 5  Retrieval      Similarity search top 3          ║
║  STEP 6  Prompting      Context + Question template      ║
║  STEP 7  Generation     google/flan-t5-small local       ║
║  STEP 8  Output         Clean answer returned            ║
║                                                          ║
║  All models run locally — no API needed!                 ║
╚══════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════╗
║           DocuMind RAG Pipeline — Complete!              ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  STEP 1  Ingestion      PyPDFLoader                      ║
║  STEP 2  Chunking       RecursiveCharacterTextSplitter   ║
║  STEP 3  Vectorization  HuggingFace all-MiniLM-L6-v2     ║
║  STEP 4  Storage        ChromaDB                         ║
║  STEP 5  Retrieval      Similarity search top 3          ║
║  STEP 6  Prompting      Context + Question template      ║
║  STEP 7  Generation     google/flan-t5-small local       ║
║  STEP 8  Output         Clean answer returned            ║
║                                                          ║
║  All models run locally — no API needed!                 ║
╚══════════════════════════════════════════════════════════╝

